### تمرین سوم
----
در این تمرین 3 فاز خواسته شده در صورت پروژه پیاده سازی شده است

In [52]:
import os, json, time, pickle, glob
import numpy as np
import torch
from torch.utils.data import DataLoader

from load_data import get_rat_hippo
from model    import RatModel, SharedRNNModel
from utils     import (collate_fn,
                       train_one_epoch, eval_one_epoch,
                       train_one_epoch_shared, eval_one_epoch_shared)

import matplotlib
matplotlib.use('Agg')          
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [53]:
RATS = {
    'achilles': {'file': 'data\\hippo_nn_achilles.npz', 'M': 120, 'T': 85, 'color': '#2979FF'},
    'buddy':    {'file': 'data\\hippo_nn_buddy.npz',    'M': 48, 'T': 58, 'color': '#00BFA5'},
    'cicero':   {'file': 'data\\hippo_nn_cicero.npz',   'M': 55, 'T': 93, 'color': '#FF6D00'},
    'gatsby':   {'file': 'data\\hippo_nn_gatsby.npz',   'M': 66, 'T': 85, 'color': '#AA00FF'},
}

SOURCE_RATS = ['achilles', 'buddy', 'cicero']
TARGET_RAT  = 'gatsby'

In [54]:
CFG = dict(
    latent_dim   = 64,
    hidden_dim   = 128,
    num_layers   = 1,
    batch_size   = 8,
    lr           = 1e-3,
    weight_decay = 1e-4,
    epochs_s1    = 60,
    epochs_s2    = 60,
    epochs_pre   = 60,
    epochs_ft    = 60,
    seed         = 42,
    max_trial_len= 500,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs('results/series1', exist_ok=True)
os.makedirs('results/series2', exist_ok=True)


print(f'  Device: {DEVICE}   PyTorch: {torch.__version__}')


  Device: cpu   PyTorch: 2.12.0+cpu


In [55]:
HIST_DIR  = 'results/series1'
MODEL_DIR = 'results/series1'
PLOT_DIR  = 'results/plots'


BG_MAIN  = '#0F1117'
BG_PANEL = '#1A1E2E'
GRID_CLR = '#252535'
SPINE_CLR= '#333344'
TICK_CLR = '#888888'
LABEL_CLR= '#AAAAAA'
CLR_GRU  = '#FF6B6B'
CLR_RNN  = '#4ECDC4'

os.makedirs('results/series3', exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

In [56]:
def make_loaders(rat_names, batch_size=8):
    train_l, val_l = {}, {}
    for rat in rat_names:
        info = RATS[rat]
        tr_ds, vl_ds = get_rat_hippo(info['file'], CFG['max_trial_len'])
        train_l[rat] = DataLoader(tr_ds, batch_size=batch_size,
                                  shuffle=True,  collate_fn=collate_fn)
        val_l[rat]   = DataLoader(vl_ds, batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_fn)
    return train_l, val_l

def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8)

def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def log_row(epoch, total, tr_loss, tr_r2, vl_loss, vl_r2, best, t0):
    print(f'  Ep {epoch:3d}/{total} | '
          f'Tr loss={tr_loss:.4f} R²={tr_r2:.4f} | '
          f'Vl loss={vl_loss:.4f} R²={vl_r2:.4f} | '
          f'Best={best:.4f} | {time.time()-t0:.0f}s')



## فاز اول
---
تعلیم مدل مستقل به ازای هر کدام از موش‌های آزمایش

In [57]:
def run_series1():
    
    print('Faze 1 — Independent model per rat')
    
    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
    summary = {}

    for rnn_type in ['RNN', 'GRU']:
        for rat, info in RATS.items():
            tag = f'{rat}_{rnn_type}'
            print(f'\n── {tag} ──────────────────────────────')
            t0 = time.time()

            tr_l, vl_l = make_loaders([rat], CFG['batch_size'])
            model = RatModel(
                input_dim=info['M'],
                latent_dim=CFG['latent_dim'],
                hidden_dim=CFG['hidden_dim'],
                rnn_type=rnn_type,
                num_layers=CFG['num_layers'],
            ).to(DEVICE)

            opt  = torch.optim.Adam(model.parameters(),
                                    lr=CFG['lr'], weight_decay=CFG['weight_decay'])
            sch  = make_scheduler(opt)
            hist = {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
            best_r2   = -np.inf
            best_path = f'results/series1/best_{tag}.pt'

            for ep in range(1, CFG['epochs_s1'] + 1):
                tr_loss, tr_r2 = train_one_epoch(model, tr_l[rat], opt, DEVICE)
                vl_loss, vl_r2 = eval_one_epoch(model,  vl_l[rat], DEVICE)
                sch.step(vl_r2)

                hist['train_loss'].append(tr_loss)
                hist['train_r2'].append(tr_r2)
                hist['val_loss'].append(vl_loss)
                hist['val_r2'].append(vl_r2)

                if vl_r2 > best_r2:
                    best_r2 = vl_r2
                    torch.save(model.state_dict(), best_path)

                if ep % 10 == 0 or ep == 1:
                    log_row(ep, CFG['epochs_s1'], tr_loss, tr_r2, vl_loss, vl_r2, best_r2, t0)

            save_json(hist, f'results/series1/history_{tag}.json')
            summary[tag] = {'best_val_r2': best_r2,
                            'final_train_r2': hist['train_r2'][-1]}
            print(f'  ✓ Best Val R² = {best_r2:.4f}')

    save_json(summary, 'results/series1/summary.json')
    print('\n  ── Series 1 Summary ──')
    for k, v in summary.items():
        print(f'  {k:20s}  Val R²={v["best_val_r2"]:.4f}')
    return summary


In [58]:
run_series1()

Faze 1 — Independent model per rat

── achilles_RNN ──────────────────────────────
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  Ep   1/60 | Tr loss=0.5180 R²=-1.0260 | Vl loss=0.3237 R²=-0.4312 | Best=-0.4312 | 0s
  Ep  10/60 | Tr loss=0.0318 R²=0.8742 | Vl loss=0.0329 R²=0.8566 | Best=0.8596 | 4s
  Ep  20/60 | Tr loss=0.0176 R²=0.9301 | Vl loss=0.0254 R²=0.8881 | Best=0.8881 | 7s
  Ep  30/60 | Tr loss=0.0122 R²=0.9514 | Vl loss=0.0241 R²=0.8919 | Best=0.8919 | 10s
  Ep  40/60 | Tr loss=0.0076 R²=0.9699 | Vl loss=0.0235 R²=0.8942 | Best=0.9009 | 13s
  Ep  50/60 | Tr loss=0.0056 R²=0.9773 | Vl loss=0.0218 R²=0.9016 | Best=0.9064 | 17s
  Ep  60/60 | Tr loss=0.0043 R²=0.9828 | Vl loss=0.0232 R²=0.8946 | Best=0.9064 | 20s
  ✓ Best Val R² = 0.9064

── buddy_RNN ──────────────────────────────
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  Ep   1/60 | Tr loss=0.5802 R²=-1.4248 | Vl loss=0.2793 R²=-0.1371 | Best=-0.1371 | 0s
  Ep  10/60 | Tr loss=0

{'achilles_RNN': {'best_val_r2': 0.9063904310266176,
  'final_train_r2': 0.9828166824041141},
 'buddy_RNN': {'best_val_r2': 0.726462185382843,
  'final_train_r2': 0.8118318145473798},
 'cicero_RNN': {'best_val_r2': 0.6859094202518463,
  'final_train_r2': 0.8144127264618873},
 'gatsby_RNN': {'best_val_r2': 0.7927055309216181,
  'final_train_r2': 0.8883007996612124},
 'achilles_GRU': {'best_val_r2': 0.9257284142076969,
  'final_train_r2': 0.9883650825358927},
 'buddy_GRU': {'best_val_r2': 0.7638026848435402,
  'final_train_r2': 0.8716895145674547},
 'cicero_GRU': {'best_val_r2': 0.7520790249109268,
  'final_train_r2': 0.8665330655872822},
 'gatsby_GRU': {'best_val_r2': 0.8037704303860664,
  'final_train_r2': 0.9276171339054903}}

In [59]:
def load_histories():
    h = {}
    for rat, _ in RATS.items():
        for rnn in ['GRU', 'RNN']:
            path = os.path.join(HIST_DIR, f'history_{rat}_{rnn}.json')
            if not os.path.exists(path):
                raise FileNotFoundError(f'History not found: {path}\n')
            h[f'{rat}_{rnn}'] = json.load(open(path))
    print(f'✓ Histories loaded ({len(h)} models)')
    return h


def collect_predictions(force_recompute=False):
    
    pkl_path = os.path.join(HIST_DIR, 'val_predictions.pkl')

    if os.path.exists(pkl_path) and not force_recompute:
        preds = pickle.load(open(pkl_path, 'rb'))
        print(f'✓ Predictions loaded from cache ({pkl_path})')
        return preds

    print('Computing predictions...')
    preds = {}
    for rnn_type in ['GRU', 'RNN']:
        for rat in RATS:
            tag = f'{rat}_{rnn_type}'
            model_path = os.path.join(MODEL_DIR, f'best_{tag}.pt')
            if not os.path.exists(model_path):
                print(f'  ⚠ Model not found: {model_path}')
                continue

            torch.manual_seed(42)
            _, val_ds = get_rat_hippo(f'data\\hippo_nn_{rat}.npz', 500)
            val_loader = DataLoader(val_ds, batch_size=32,
                                    shuffle=False, collate_fn=collate_fn)

            M = RATS[rat]['M']
            model = RatModel(M, 64, 128, rnn_type)
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.eval()

            all_pred, all_true = [], []
            with torch.no_grad():
                for X_pad, Y_pad, mask, lengths in val_loader:
                    pred = model(X_pad, lengths)
                    all_pred.extend(pred[mask].numpy().tolist())
                    all_true.extend(Y_pad[mask].numpy().tolist())

            preds[tag] = {
                'pred': np.array(all_pred),
                'true': np.array(all_true),
            }
            print(f'  {tag}: {len(all_pred):,} timesteps')

    pickle.dump(preds, open(pkl_path, 'wb'))
    print(f'✓ Predictions saved to {pkl_path}')
    return preds

In [60]:
def style_ax(ax):
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors=TICK_CLR, labelsize=8)
    ax.xaxis.label.set_color(LABEL_CLR)
    ax.yaxis.label.set_color(LABEL_CLR)
    ax.grid(color=GRID_CLR, lw=0.5)
    for sp in ax.spines.values():
        sp.set_color(SPINE_CLR)

def r2_rmse(pred, true):
    pred = np.asarray(pred, dtype=np.float32)
    true = np.asarray(true, dtype=np.float32)
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    r2   = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    rmse = np.sqrt(np.mean((true - pred) ** 2))
    return r2, rmse


In [61]:
def plot_gru_curves(histories):
    fig = plt.figure(figsize=(20, 9))
    fig.patch.set_facecolor(BG_MAIN)
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.30)
    fig.suptitle('Series 1 — GRU: Training Curves',
                 fontsize=16, fontweight='bold', color='white', y=0.98)

    for col, rat in enumerate(RATS):
        h   = histories[f'{rat}_GRU']
        eps = range(1, len(h['train_loss']) + 1)
        c   = RATS[rat]['color']

        ax = fig.add_subplot(gs[0, col])
        style_ax(ax)
        ax.plot(eps, h['train_loss'], color=CLR_GRU, lw=1.5, label='Train')
        ax.plot(eps, h['val_loss'],   color=CLR_RNN, lw=1.5, label='Val', ls='--')
        best_ep = int(np.argmin(h['val_loss'])) + 1
        ax.axvline(best_ep, color='white', lw=0.8, ls=':', alpha=0.45)
        ax.set_title(rat.capitalize(), color=c, fontsize=13,
                     fontweight='bold', pad=6)
        ax.set_xlabel('Epoch');  ax.set_ylabel('MSE Loss')
        if col == 0:
            ax.legend(fontsize=8, framealpha=0.2, labelcolor='white')

        ax2 = fig.add_subplot(gs[1, col])
        style_ax(ax2)
        ax2.plot(eps, h['train_r2'], color=CLR_GRU, lw=1.5, label='Train')
        ax2.plot(eps, h['val_r2'],   color=CLR_RNN, lw=1.5, label='Val', ls='--')
        best_r2 = max(h['val_r2'])
        best_ep2 = int(np.argmax(h['val_r2'])) + 1
        ax2.axvline(best_ep2, color='white', lw=0.8, ls=':', alpha=0.45)
        ax2.axhline(best_r2,  color=c,       lw=0.8, ls=':', alpha=0.60)
        ax2.annotate(
            f'R²={best_r2:.3f}',
            xy=(best_ep2, best_r2),
            xytext=(best_ep2 + 4, best_r2 - 0.10),
            color=c, fontsize=8,
            arrowprops=dict(arrowstyle='->', color=c, lw=0.8),
        )
        ax2.set_xlabel('Epoch');  ax2.set_ylabel('R²')
        ax2.set_ylim([-0.3, 1.05])
        if col == 0:
            ax2.legend(fontsize=8, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p1_gru_curves.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


In [62]:
def plot_rnn_vs_gru(histories):
    fig = plt.figure(figsize=(20, 9))
    fig.patch.set_facecolor(BG_MAIN)
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.30)
    fig.suptitle('Series 1 — RNN vs GRU: Validation & Train R²',
                 fontsize=16, fontweight='bold', color='white', y=0.98)

    for col, rat in enumerate(RATS):
        h_g = histories[f'{rat}_GRU']
        h_r = histories[f'{rat}_RNN']
        eps = list(range(1, 61))
        c   = RATS[rat]['color']

        ax = fig.add_subplot(gs[0, col])
        style_ax(ax)
        ax.plot(eps, h_g['val_r2'], color=CLR_GRU, lw=2, label='GRU')
        ax.plot(eps, h_r['val_r2'], color=CLR_RNN, lw=2, label='RNN', ls='--')
        
        ax.fill_between(
            eps, h_r['val_r2'], h_g['val_r2'],
            where=[g > r for g, r in zip(h_g['val_r2'], h_r['val_r2'])],
            alpha=0.18, color=CLR_GRU, label='GRU advantage',
        )
        ax.set_title(rat.capitalize(), color=c, fontsize=13,
                     fontweight='bold', pad=6)
        ax.set_xlabel('Epoch');  ax.set_ylabel('Val R²')
        ax.set_ylim([-0.10, 1.05])
        ax.legend(fontsize=8, framealpha=0.2, labelcolor='white')

        ax2 = fig.add_subplot(gs[1, col])
        style_ax(ax2)
        ax2.plot(eps, h_g['train_r2'], color=CLR_GRU, lw=2, label='GRU')
        ax2.plot(eps, h_r['train_r2'], color=CLR_RNN, lw=2, label='RNN', ls='--')
        ax2.set_xlabel('Epoch');  ax2.set_ylabel('Train R²')
        ax2.set_ylim([-0.10, 1.05])
        ax2.legend(fontsize=8, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p2_rnn_vs_gru.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


In [63]:
def plot_comparison_table(histories):
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Final Performance Comparison',
                 fontsize=16, fontweight='bold', color='white')

    ax  = axes[0]
    ax.set_facecolor(BG_PANEL)
    x   = np.arange(len(RATS))
    w   = 0.35

    gru_val   = [max(histories[f'{r}_GRU']['val_r2'])   for r in RATS]
    rnn_val   = [max(histories[f'{r}_RNN']['val_r2'])   for r in RATS]

    bars1 = ax.bar(x - w/2, rnn_val, w, label='RNN  (Val R²)',
                   color=CLR_RNN, alpha=0.85, zorder=3)
    bars2 = ax.bar(x + w/2, gru_val, w, label='GRU  (Val R²)',
                   color=CLR_GRU, alpha=0.85, zorder=3)

    for bar, val in zip(bars1, rnn_val):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=CLR_RNN, fontweight='bold')
    for bar, val in zip(bars2, gru_val):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=CLR_GRU, fontweight='bold')

    for ref in [0.7, 0.8, 0.9]:
        ax.axhline(ref, color='white', lw=0.7, ls=':', alpha=0.25)

    ax.set_xticks(x)
    ax.set_xticklabels([r.capitalize() for r in RATS],
                       fontsize=12, color='white')
    ax.set_ylabel('Best Validation R²', color=LABEL_CLR, fontsize=11)
    ax.set_ylim([0.50, 1.05])
    ax.tick_params(colors=TICK_CLR)
    for sp in ax.spines.values(): sp.set_color(SPINE_CLR)
    ax.grid(axis='y', color=GRID_CLR, lw=0.5, zorder=0)
    ax.legend(fontsize=10, framealpha=0.2, labelcolor='white')
    ax.set_title('Best Validation R²: RNN vs GRU',
                 color='white', fontsize=12, pad=10)

    ax2 = axes[1]
    ax2.set_facecolor(BG_MAIN)
    ax2.axis('off')

    headers = ['Rat', 'Neurons', 'Trials',
               'RNN Val R²', 'GRU Val R²', 'Δ(GRU−RNN)',
               'RNN Train R²', 'GRU Train R²']
    rows = []
    for rat in RATS:
        g_v = max(histories[f'{rat}_GRU']['val_r2'])
        r_v = max(histories[f'{rat}_RNN']['val_r2'])
        g_t = max(histories[f'{rat}_GRU']['train_r2'])
        r_t = max(histories[f'{rat}_RNN']['train_r2'])
        delta = g_v - r_v
        rows.append([
            rat.capitalize(),
            str(RATS[rat]['M']),
            str(RATS[rat]['T']),
            f'{r_v:.4f}',
            f'{g_v:.4f}',
            f'+{delta:.4f}' if delta >= 0 else f'{delta:.4f}',
            f'{r_t:.4f}',
            f'{g_t:.4f}',
        ])

    tbl = ax2.table(cellText=rows, colLabels=headers,
                    cellLoc='center', loc='center',
                    bbox=[0, 0.10, 1, 0.85])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(SPINE_CLR)
        if r == 0:                          
            cell.set_facecolor('#2A2E45')
            cell.get_text().set_color('white')
            cell.get_text().set_fontweight('bold')
        elif r % 2 == 0:
            cell.set_facecolor('#1E2235')
            cell.get_text().set_color('#CCCCCC')
        else:
            cell.set_facecolor('#171B2C')
            cell.get_text().set_color('#CCCCCC')
        if r > 0 and c == 5:                
            v = float(rows[r - 1][5])
            cell.get_text().set_color('#66FF99' if v >= 0 else '#FF6666')

    ax2.set_title('Detailed Statistics', color='white', fontsize=12, pad=10)

    out = os.path.join(PLOT_DIR, 'p3_comparison_table.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


In [64]:
def plot_scatter(preds):
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Predicted vs True Position  (Validation Set)',
                 fontsize=16, fontweight='bold', color='white', y=0.99)

    for col, rat in enumerate(RATS):
        c = RATS[rat]['color']
        for row, rnn in enumerate(['GRU', 'RNN']):
            ax  = axes[row][col]
            ax.set_facecolor(BG_PANEL)
            tag = f'{rat}_{rnn}'
            p   = preds[tag]['pred']
            t   = preds[tag]['true']

            ax.hexbin(t, p, gridsize=40, cmap='plasma',
                      mincnt=1, extent=[0, 1.6, 0, 1.6])
            ax.plot([0, 1.6], [0, 1.6], 'w--', lw=1.2, alpha=0.7)

            r2, rmse = r2_rmse(p, t)
            ax.text(0.05, 1.48,
                    f'R²={r2:.4f}\nRMSE={rmse:.4f} m',
                    color='white', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#000000AA', edgecolor='none'))

            ax.set_title(f'{rat.capitalize()} — {rnn}',
                         color=c, fontsize=12, fontweight='bold')
            ax.set_xlabel('True position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_ylabel('Predicted position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_xlim([0, 1.6]);  ax.set_ylim([0, 1.6])
            ax.tick_params(colors=TICK_CLR, labelsize=8)
            for sp in ax.spines.values(): sp.set_color(SPINE_CLR)

    out = os.path.join(PLOT_DIR, 'p4_scatter.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


In [65]:
def plot_error_dist(preds):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle('Series 1 — Prediction Error Distribution  (GRU vs RNN)',
                 fontsize=15, fontweight='bold', color='white', y=1.02)

    for col, rat in enumerate(RATS):
        ax = axes[col]
        style_ax(ax)
        c  = RATS[rat]['color']

        for rnn, clr in [('GRU', CLR_GRU), ('RNN', CLR_RNN)]:
            p   = np.asarray(preds[f'{rat}_{rnn}']['pred'], dtype=np.float32)
            t   = np.asarray(preds[f'{rat}_{rnn}']['true'], dtype=np.float32)
            err = p - t
            ax.hist(err, bins=60, alpha=0.55, color=clr, density=True,
                    label=f'{rnn}  σ={err.std():.3f}m')
            ax.axvline(err.mean(), color=clr, lw=1.5, ls=':')

        ax.axvline(0, color='white', lw=1.0, ls='--', alpha=0.45)
        ax.set_title(rat.capitalize(), color=c, fontsize=13, fontweight='bold')
        ax.set_xlabel('Error  (pred − true)  [m]', color=LABEL_CLR, fontsize=10)
        ax.set_ylabel('Density', color=LABEL_CLR, fontsize=10)
        ax.legend(fontsize=9, framealpha=0.2, labelcolor='white')

    out = os.path.join(PLOT_DIR, 'p5_error_dist.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')


In [66]:
def plot_timeseries(preds, n_timesteps=250):
    fig, axes = plt.subplots(4, 2, figsize=(18, 14))
    fig.patch.set_facecolor(BG_MAIN)
    fig.suptitle(f'Series 1 — Sample Trajectory: Predicted vs True  '
                 f'(first {n_timesteps} timesteps of validation)',
                 fontsize=15, fontweight='bold', color='white', y=1.0)

    for row, rat in enumerate(RATS):
        c = RATS[rat]['color']
        for col, rnn in enumerate(['GRU', 'RNN']):
            ax  = axes[row][col]
            style_ax(ax)
            tag = f'{rat}_{rnn}'
            p   = preds[tag]['pred']
            t   = preds[tag]['true']
            N   = min(n_timesteps, len(p))
            t_ax = np.arange(N)

            ax.plot(t_ax, t[:N], color='white', lw=1.8,
                    alpha=0.9, label='True position')
            ax.plot(t_ax, p[:N], color=c, lw=1.5,
                    alpha=0.85, label=f'{rnn} predicted')
            ax.fill_between(t_ax, t[:N], p[:N], alpha=0.18, color=c)

            r2, rmse = r2_rmse(p, t)
            ax.text(0.02, 0.06, f'R²={r2:.4f}',
                    transform=ax.transAxes, color=c,
                    fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.25',
                              facecolor='#000000AA', edgecolor='none'))

            ax.set_ylim([0, 1.6])
            ax.set_xlabel('Timestep', color=LABEL_CLR, fontsize=9)
            ax.set_ylabel('Position (m)', color=LABEL_CLR, fontsize=9)
            ax.set_title(f'{rat.capitalize()} — {rnn}',
                         color=c, fontsize=11, fontweight='bold')
            ax.legend(fontsize=8, framealpha=0.2, labelcolor='white',
                      loc='upper right')

    out = os.path.join(PLOT_DIR, 'p6_timeseries.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG_MAIN)
    plt.close()
    print(f'  ✓ {out}')

In [67]:
print('═' * 52)
print('  Series 1 — Visualization')
print('═' * 52 + '\n')

histories = load_histories()
preds     = collect_predictions(force_recompute=False)

print('\nGenerating plots...')
plot_gru_curves(histories)        
plot_rnn_vs_gru(histories)        
plot_comparison_table(histories) 
plot_scatter(preds)               
plot_error_dist(preds)            
plot_timeseries(preds)            

print(f'\n✓ All 6 plots saved to: {PLOT_DIR}/')
print('\nفایل‌های خروجی:')
for i, name in enumerate([
    'p1_gru_curves.png       ← Loss و R² در طول آموزش (GRU)',
    'p2_rnn_vs_gru.png       ← مقایسه RNN و GRU',
    'p3_comparison_table.png ← جدول + bar chart نهایی',
    'p4_scatter.png          ← Predicted vs True (scatter)',
    'p5_error_dist.png       ← توزیع خطا',
    'p6_timeseries.png       ← پیش‌بینی در طول زمان',
], 1):
    print(f'  {i}. {name}')

════════════════════════════════════════════════════
  Series 1 — Visualization
════════════════════════════════════════════════════

✓ Histories loaded (8 models)
Computing predictions...
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  achilles_GRU: 2,134 timesteps
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  buddy_GRU: 1,457 timesteps
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  cicero_GRU: 5,847 timesteps
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  gatsby_GRU: 3,521 timesteps
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  achilles_RNN: 2,134 timesteps
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  buddy_RNN: 1,457 timesteps
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  cicero_RNN: 5,847 timesteps
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  gatsby_RNN: 3,521 timesteps
✓ Predictions saved to results/series

In [68]:
def run_series2():
    print('=' * 55)
    print('  SERIES 2 — Shared RNN ')
    print('=' * 55)

    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
    t0 = time.time()

    print('\nLoading datasets...')
    tr_l, vl_l = make_loaders(list(RATS.keys()), CFG['batch_size'])

    rat_dims = {r: RATS[r]['M'] for r in RATS}
    model = SharedRNNModel(
        rat_dims=rat_dims,
        latent_dim=CFG['latent_dim'],
        hidden_dim=CFG['hidden_dim'],
        rnn_type='GRU',
        num_layers=CFG['num_layers'],
    ).to(DEVICE)
    print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

    opt  = torch.optim.Adam(model.parameters(),
                            lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch  = make_scheduler(opt)
    hist = {r: {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
            for r in RATS}
    best_avg = -np.inf
    best_path = 'results/series2/best_shared.pt'

    for ep in range(1, CFG['epochs_s2'] + 1):
        tr_stats = train_one_epoch_shared(model, tr_l, opt, DEVICE)
        vl_stats = eval_one_epoch_shared(model, vl_l, DEVICE)

        avg_vl = 0.0
        for rat in RATS:
            hist[rat]['train_loss'].append(tr_stats[rat][0])
            hist[rat]['train_r2'].append(tr_stats[rat][1])
            hist[rat]['val_loss'].append(vl_stats[rat][0])
            hist[rat]['val_r2'].append(vl_stats[rat][1])
            avg_vl += vl_stats[rat][1]
        avg_vl /= len(RATS)
        sch.step(avg_vl)

        if avg_vl > best_avg:
            best_avg = avg_vl
            torch.save(model.state_dict(), best_path)

        if ep % 10 == 0 or ep == 1:
            print(f'\n  Epoch {ep:3d}/{CFG["epochs_s2"]}  (t={time.time()-t0:.0f}s)')
            for rat in RATS:
                print(f'    {rat:8s}: Tr R²={tr_stats[rat][1]:.4f}  Vl R²={vl_stats[rat][1]:.4f}')
            print(f'    AvgVl R²={avg_vl:.4f}  Best={best_avg:.4f}')

    save_json(hist, 'results/series2/history_shared.json')
    summary = {r: {'best_val_r2': max(hist[r]['val_r2']),
                   'final_train_r2': hist[r]['train_r2'][-1]} for r in RATS}
    save_json(summary, 'results/series2/summary.json')

    print('\n  ── Series 2 Summary ──')
    for r, s in summary.items():
        print(f'  {r:8s}: Val R²={s["best_val_r2"]:.4f}')
    return model, hist, summary

In [69]:
s2_model, s2_hist, s2_summary = run_series2()

  SERIES 2 — Shared RNN 

Loading datasets...
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  Parameters: 128,831

  Epoch   1/60  (t=6s)
    achilles: Tr R²=-1.1647  Vl R²=-0.1252
    buddy   : Tr R²=-2.0027  Vl R²=-0.1387
    cicero  : Tr R²=-1.0775  Vl R²=-0.1008
    gatsby  : Tr R²=-1.2903  Vl R²=-0.2652
    AvgVl R²=-0.1575  Best=-0.1575

  Epoch  10/60  (t=60s)
    achilles: Tr R²=0.8392  Vl R²=0.7779
    buddy   : Tr R²=0.5239  Vl R²=0.5469
    cicero  : Tr R²=0.5349  Vl R²=0.4898
    gatsby  : Tr R²=0.5659  Vl R²=0.4133
    AvgVl R²=0.5570  Best=0.5570

  Epoch  20/60  (t=128s)
    achilles: Tr R²=0.9122  Vl R²=0.8881
    buddy   : Tr R²=0.7345  Vl R²=0.6887
    cicero  : Tr R²=0.7866  Vl R²=0.7238
    gatsby  : Tr R²=0.8131  Vl R²=0.7428
    AvgVl R²=0.7608  Be

In [70]:
for r, s in s2_summary.items():
    print(f'    {r:8s} Val R²={s["best_val_r2"]:.4f}')

    achilles Val R²=0.9324
    buddy    Val R²=0.7812
    cicero   Val R²=0.8020
    gatsby   Val R²=0.7934


In [71]:
def load_series2_model():
    rat_dims = {r: RATS[r]['M'] for r in RATS}

    model = SharedRNNModel(
        rat_dims=rat_dims,
        latent_dim=CFG['latent_dim'],
        hidden_dim=CFG['hidden_dim'],
        rnn_type='GRU',
        num_layers=CFG['num_layers'],
    ).to(DEVICE)

    model.load_state_dict(torch.load('results/series2/best_shared.pt', map_location=DEVICE))
    model.eval()
    return model

def collect_predictions_series2(model=None, split='val'):
    
    if model is None:
        model = load_series2_model()

    preds = {}

    for rat in RATS:
        tr_ds, vl_ds = get_rat_hippo(RATS[rat]['file'], CFG['max_trial_len'])
        ds = vl_ds if split == 'val' else tr_ds

        loader = DataLoader(
            ds,
            batch_size=32,
            shuffle=False,
            collate_fn=collate_fn
        )

        all_pred = []
        all_true = []
        trials = []

        with torch.no_grad():
            for X_pad, Y_pad, mask, lengths in loader:
                X_pad = X_pad.to(DEVICE)

                pred = model(X_pad, lengths, rat).cpu()

                for i in range(X_pad.shape[0]):
                    L = lengths[i]
                    p_i = pred[i, :L].numpy()
                    t_i = Y_pad[i, :L].numpy()

                    all_pred.extend(p_i)
                    all_true.extend(t_i)

                    trials.append({
                        'pred': p_i,
                        'true': t_i
                    })

        preds[rat] = {
            'pred': np.asarray(all_pred, dtype=np.float32),
            'true': np.asarray(all_true, dtype=np.float32),
            'trials': trials
        }

    return preds


In [72]:
def plot_series2_learning_curves(hist=None):
    if hist is None:
        hist = load_json('results/series2/history_shared.json')

    rats = list(RATS.keys())
    fig, axes = plt.subplots(len(rats), 2, figsize=(12, 3.5 * len(rats)))
    if len(rats) == 1:
        axes = np.array([axes])

    for i, rat in enumerate(rats):
        ep = np.arange(1, len(hist[rat]['train_loss']) + 1)

        # Loss
        ax = axes[i, 0]
        ax.plot(ep, hist[rat]['train_loss'], label='Train Loss', lw=2)
        ax.plot(ep, hist[rat]['val_loss'], label='Val Loss', lw=2)
        ax.set_title(f'{rat} — Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE')
        ax.grid(alpha=0.3)
        ax.legend()

        # R2
        ax = axes[i, 1]
        ax.plot(ep, hist[rat]['train_r2'], label='Train R²', lw=2)
        ax.plot(ep, hist[rat]['val_r2'], label='Val R²', lw=2)
        ax.set_title(f'{rat} — R²')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('R²')
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    out = 'results/plots/s2_learning_curves.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.close()
    print('saved →', out)


def plot_series2_summary(summary=None):
    if summary is None:
        summary = load_json('results/series2/summary.json')

    rats = list(RATS.keys())
    vals = [summary[r]['best_val_r2'] for r in rats]
    avg = np.mean(vals)

    plt.figure(figsize=(8, 5))
    bars = plt.bar(rats, vals)
    plt.axhline(avg, color='red', ls='--', lw=2, label=f'Average = {avg:.3f}')

    for b, v in zip(bars, vals):
        plt.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=10)

    plt.ylabel('Best Validation R²')
    plt.title('Series 2 — Shared Model Performance per Rat')
    plt.ylim(0, max(vals) + 0.12 if len(vals) else 1.0)
    plt.grid(axis='y', alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out = 'results/plots/s2_summary_bar.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.close()
    print('saved →', out)

def plot_series2_scatter(preds=None):
    if preds is None:
        preds = collect_predictions_series2()

    rats = list(RATS.keys())
    fig, axes = plt.subplots(1, len(rats), figsize=(4.5 * len(rats), 4.2))
    if len(rats) == 1:
        axes = [axes]

    for ax, rat in zip(axes, rats):
        p = preds[rat]['pred']
        t = preds[rat]['true']
        r2, rmse = r2_rmse(p, t)

        hb = ax.hexbin(t, p, gridsize=45, cmap='viridis', mincnt=1)
        ax.plot([0, 1.6], [0, 1.6], 'r--', lw=1.5)

        ax.set_title(f'{rat}\nR²={r2:.3f}, RMSE={rmse:.3f}')
        ax.set_xlabel('True Position')
        ax.set_ylabel('Predicted Position')
        ax.set_xlim(0, 1.6)
        ax.set_ylim(0, 1.6)

    fig.colorbar(hb, ax=axes, shrink=0.85, label='Count')
    plt.tight_layout()

    out = 'results/plots/s2_scatter_hexbin.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.close()
    print('saved →', out)




In [73]:
def plot_series2_error_distribution(preds=None):
    if preds is None:
        preds = collect_predictions_series2()

    rats = list(RATS.keys())
    fig, axes = plt.subplots(1, len(rats), figsize=(4.5 * len(rats), 4.0))
    if len(rats) == 1:
        axes = [axes]

    for ax, rat in zip(axes, rats):
        p = preds[rat]['pred']
        t = preds[rat]['true']
        err = p - t

        ax.hist(err, bins=50, alpha=0.85)
        ax.axvline(0, color='red', ls='--', lw=1.5)

        ax.set_title(f'{rat}\nmean={err.mean():.3f}, std={err.std():.3f}')
        ax.set_xlabel('Prediction Error')
        ax.set_ylabel('Count')
        ax.grid(alpha=0.25)

    plt.tight_layout()
    out = 'results/plots/s2_error_distribution.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.close()
    print('saved →', out)

In [74]:
import random

def plot_series2_timeseries(preds=None, n_trials=3, seed=42):
    if preds is None:
        preds = collect_predictions_series2()

    random.seed(seed)
    rats = list(RATS.keys())

    fig, axes = plt.subplots(len(rats), n_trials, figsize=(4.5*n_trials, 3.2*len(rats)))
    if len(rats) == 1:
        axes = np.expand_dims(axes, axis=0)
    if n_trials == 1:
        axes = np.expand_dims(axes, axis=1)

    for i, rat in enumerate(rats):
        trials = preds[rat]['trials']
        idxs = random.sample(range(len(trials)), min(n_trials, len(trials)))

        for j in range(n_trials):
            ax = axes[i, j]
            if j < len(idxs):
                tr = trials[idxs[j]]
                ax.plot(tr['true'], label='True', lw=2)
                ax.plot(tr['pred'], label='Pred', lw=2)
                r2, rmse = r2_rmse(tr['pred'], tr['true'])
                ax.set_title(f'{rat} | trial {idxs[j]}\nR²={r2:.3f}')
                ax.set_xlabel('Time')
                ax.set_ylabel('Position')
                ax.set_ylim(0, 1.6)
                ax.grid(alpha=0.3)
                if i == 0 and j == 0:
                    ax.legend()
            else:
                ax.axis('off')

    plt.tight_layout()
    out = 'results/plots/s2_timeseries.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.close()
    print('saved →', out)


In [75]:
def run_series2_visualization():
    print('=' * 60)
    print('Series 2 Visualization')
    print('=' * 60)

    hist = load_json('results/series2/history_shared.json')
    summary = load_json('results/series2/summary.json')
    model = load_series2_model()
    preds = collect_predictions_series2(model=model, split='val')

    plot_series2_learning_curves(hist)
    plot_series2_summary(summary)
    plot_series2_scatter(preds)
    plot_series2_error_distribution(preds)
    plot_series2_timeseries(preds, n_trials=3)

    print('\nAll Series 2 plots saved in results/plots/')


In [76]:
run_series2_visualization()

Series 2 Visualization
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
saved → results/plots/s2_learning_curves.png
saved → results/plots/s2_summary_bar.png


C:\Users\M A T I N\AppData\Local\Temp\ipykernel_19364\2072581134.py:92: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


saved → results/plots/s2_scatter_hexbin.png
saved → results/plots/s2_error_distribution.png
saved → results/plots/s2_timeseries.png

All Series 2 plots saved in results/plots/


In [91]:
def run_series3():
    print('=' * 55)
    print('  SERIES 3 — Transfer Learning')
    print('=' * 55)

    torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])

    print('model has been trained on :')
    for it in SOURCE_RATS:
        print(it)
    
    t0 = time.time()

    src_info = {r: RATS[r] for r in SOURCE_RATS}
    tr_l, vl_l = make_loaders(SOURCE_RATS, CFG['batch_size'])

    rat_dims = {r: RATS[r]['M'] for r in SOURCE_RATS}
    model = SharedRNNModel(
        rat_dims=rat_dims,
        latent_dim=CFG['latent_dim'],
        hidden_dim=CFG['hidden_dim'],
        rnn_type='GRU',
        num_layers=CFG['num_layers'],
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(),
                           lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch = make_scheduler(opt)
    pre_hist = {r: {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
                for r in SOURCE_RATS}
    best_pre = -np.inf
    pre_path = 'results/series3/pretrained.pt'

    for ep in range(1, CFG['epochs_pre'] + 1):
        tr_s = train_one_epoch_shared(model, tr_l, opt, DEVICE)
        vl_s = eval_one_epoch_shared(model, vl_l, DEVICE)
        avg  = 0.0
        for r in SOURCE_RATS:
            pre_hist[r]['train_loss'].append(tr_s[r][0])
            pre_hist[r]['train_r2'].append(tr_s[r][1])
            pre_hist[r]['val_loss'].append(vl_s[r][0])
            pre_hist[r]['val_r2'].append(vl_s[r][1])
            avg += vl_s[r][1]
        avg /= len(SOURCE_RATS)
        sch.step(avg)
        if avg > best_pre:
            best_pre = avg
            torch.save(model.state_dict(), pre_path)
        if ep % 10 == 0 or ep == 1:
            print(f'  Ep {ep:3d}/{CFG["epochs_pre"]}  AvgVl R²={avg:.4f}  Best={best_pre:.4f}  t={time.time()-t0:.0f}s')

    save_json(pre_hist, 'results/series3/history_pretrain.json')
    print(f'  ✓ Pre-trained. Best Avg Val R²={best_pre:.4f}')

    print(f'\n  Phase 2: Fine-tuning new Encoder for "{TARGET_RAT}"')
    t1 = time.time()

    model.load_state_dict(torch.load(pre_path, map_location=DEVICE))

    model.freeze_shared()

    model.add_rat_encoder(TARGET_RAT, RATS[TARGET_RAT]['M'])
    model = model.to(DEVICE)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_trainable:,} / Total: {n_total:,}')

    tgt_tr, tgt_vl = make_loaders([TARGET_RAT], CFG['batch_size'])
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt2 = torch.optim.Adam(trainable_params,
                            lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sch2 = make_scheduler(opt2)

    ft_hist = {'train_loss':[], 'train_r2':[], 'val_loss':[], 'val_r2':[]}
    best_ft = -np.inf
    ft_path = 'results/series3/finetuned.pt'

    for ep in range(1, CFG['epochs_ft'] + 1):
        tr_s = train_one_epoch_shared(model, tgt_tr, opt2, DEVICE)
        vl_s = eval_one_epoch_shared(model, tgt_vl, DEVICE)
        tr_loss, tr_r2 = tr_s[TARGET_RAT]
        vl_loss, vl_r2 = vl_s[TARGET_RAT]
        sch2.step(vl_r2)

        ft_hist['train_loss'].append(tr_loss)
        ft_hist['train_r2'].append(tr_r2)
        ft_hist['val_loss'].append(vl_loss)
        ft_hist['val_r2'].append(vl_r2)

        if vl_r2 > best_ft:
            best_ft = vl_r2
            torch.save(model.state_dict(), ft_path)

        if ep % 10 == 0 or ep == 1:
            log_row(ep, CFG['epochs_ft'], tr_loss, tr_r2, vl_loss, vl_r2, best_ft, t1)

    save_json(ft_hist, 'results/series3/history_finetune.json')
    summary = {
        'pretrain': {r: {'best_val_r2': max(pre_hist[r]['val_r2'])} for r in SOURCE_RATS},
        'finetune_gatsby': {'best_val_r2': best_ft,
                            'final_train_r2': ft_hist['train_r2'][-1]}
    }
    save_json(summary, 'results/series3/summary.json')

    print(f'  ✓ Fine-tuned gatsby. Best Val R²={best_ft:.4f}')
    return model, ft_hist, summary


In [92]:
model3, ft_hist, s3_summary = run_series3()

  SERIES 3 — Transfer Learning
model has been trained on :
achilles
buddy
cicero
  [data\hippo_nn_achilles.npz] trials=85 neurons=120 train=68 val=17
  [data\hippo_nn_buddy.npz] trials=58 neurons=48 train=46 val=12
  [data\hippo_nn_cicero.npz] trials=93 neurons=55 train=74 val=19
  Ep   1/60  AvgVl R²=-0.1804  Best=-0.1804  t=5s
  Ep  10/60  AvgVl R²=0.7107  Best=0.7107  t=51s
  Ep  20/60  AvgVl R²=0.7632  Best=0.7632  t=95s
  Ep  30/60  AvgVl R²=0.7768  Best=0.7768  t=139s
  Ep  40/60  AvgVl R²=0.8083  Best=0.8083  t=182s
  Ep  50/60  AvgVl R²=0.8419  Best=0.8419  t=227s
  Ep  60/60  AvgVl R²=0.8380  Best=0.8554  t=273s
  ✓ Pre-trained. Best Avg Val R²=0.8554

  Phase 2: Fine-tuning new Encoder for "gatsby"
  Trainable: 46,014 / Total: 128,831
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
  Ep   1/60 | Tr loss=0.3069 R²=-0.1792 | Vl loss=0.3203 R²=-0.1296 | Best=-0.1296 | 1s
  Ep  10/60 | Tr loss=0.0565 R²=0.7837 | Vl loss=0.0886 R²=0.6939 | Best=0.6959 | 13s
  Ep 

In [80]:
print(f'    {TARGET_RAT} finetune  Val R²={s3_summary["finetune_gatsby"]["best_val_r2"]:.4f}')

    gatsby finetune  Val R²=0.7974


In [81]:
def plot_series3_training_curves():
    hist = load_json('results/series3/history_finetune.json')

    epochs = np.arange(1, len(hist['train_loss']) + 1)

    fig, ax = plt.subplots(1,3, figsize=(15,4))

    # Loss
    ax[0].plot(epochs, hist['train_loss'], lw=2, label='Train')
    ax[0].plot(epochs, hist['val_loss'], lw=2, label='Test')
    ax[0].set_title('Series3 — Loss (gatsby)')
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('MSE')
    ax[0].grid(alpha=0.3)
    ax[0].legend()

    # Train R2
    ax[1].plot(epochs, hist['train_r2'], lw=2, color='tab:green')
    ax[1].set_title('Train R² (gatsby)')
    ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('R²')
    ax[1].grid(alpha=0.3)

    # Test R2
    ax[2].plot(epochs, hist['val_r2'], lw=2, color='tab:orange')
    ax[2].set_title('Test R² (gatsby)')
    ax[2].set_xlabel('Epoch')
    ax[2].set_ylabel('R²')
    ax[2].grid(alpha=0.3)

    plt.tight_layout()
    path = 'results/plots/s3_training_curves.png'
    plt.savefig(path, dpi=160)
    plt.close()

    print('saved →', path)


In [82]:
def plot_series3_scatter(model=None):

    if model is None:
        rat_dims = {r: RATS[r]['M'] for r in SOURCE_RATS}
        rat_dims[TARGET_RAT] = RATS[TARGET_RAT]['M']

        model = SharedRNNModel(
            rat_dims=rat_dims,
            latent_dim=CFG['latent_dim'],
            hidden_dim=CFG['hidden_dim'],
            rnn_type='GRU',
            num_layers=CFG['num_layers']
        ).to(DEVICE)

        model.load_state_dict(
            torch.load('results/series3/finetuned.pt', map_location=DEVICE)
        )
        model.eval()

    tr_ds, vl_ds = get_rat_hippo(RATS[TARGET_RAT]['file'], CFG['max_trial_len'])

    loader = DataLoader(
        vl_ds,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn
    )

    preds=[]
    trues=[]

    with torch.no_grad():
        for X_pad,Y_pad,mask,lengths in loader:

            X_pad = X_pad.to(DEVICE)
            out = model(X_pad,lengths,TARGET_RAT).cpu()

            for i,L in enumerate(lengths):
                preds.extend(out[i,:L].numpy())
                trues.extend(Y_pad[i,:L].numpy())

    preds=np.array(preds)
    trues=np.array(trues)

    r2,rmse = r2_rmse(preds,trues)

    plt.figure(figsize=(5,5))

    hb=plt.hexbin(trues,preds,
                  gridsize=45,
                  cmap='viridis',
                  mincnt=1)

    plt.plot([0,1.6],[0,1.6],'r--')

    plt.title(f'Series3 Transfer — gatsby\nR²={r2:.3f}  RMSE={rmse:.3f}')
    plt.xlabel('True position')
    plt.ylabel('Predicted position')

    plt.xlim(0,1.6)
    plt.ylim(0,1.6)

    plt.colorbar(hb,label='count')

    path='results/plots/s3_scatter_gatsby.png'
    plt.tight_layout()
    plt.savefig(path,dpi=160)
    plt.close()

    print('saved →',path)


In [ ]:
def collect_predictions_series3(target_rat=TARGET_RAT, model=model3, split='val'):

    if model == None:
        raise Exception("mode is None")

    tr_ds, vl_ds = get_rat_hippo(
        RATS[target_rat]['file'],
        CFG['max_trial_len']
    )

    ds = vl_ds if split == 'val' else tr_ds

    loader = DataLoader(
        ds,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn
    )

    preds = []
    trues = []
    trials = []

    with torch.no_grad():

        for X_pad, Y_pad, mask, lengths in loader:

            X_pad = X_pad.to(DEVICE)

            out = model(X_pad, lengths, target_rat).cpu()

            for i, L in enumerate(lengths):

                p = out[i, :L].numpy()
                t = Y_pad[i, :L].numpy()

                preds.extend(p)
                trues.extend(t)

                trials.append({
                    'pred': p,
                    'true': t
                })

    preds = np.asarray(preds, dtype=np.float32)
    trues = np.asarray(trues, dtype=np.float32)

    return {
        'pred': preds,
        'true': trues,
        'trials': trials
    }


In [93]:
def plot_series3_timeseries(n_trials=4, model = model3):

    preds = collect_predictions_series3(TARGET_RAT, model = model)

    trials = preds['trials']
    idx = np.random.choice(len(trials), n_trials, replace=False)

    _ ,ax = plt.subplots(n_trials,1,figsize=(10,3*n_trials))

    if n_trials==1:
        ax=[ax]

    for i,t in enumerate(idx):

        tr = trials[t]

        r2,rmse = r2_rmse(tr['pred'],tr['true'])

        ax[i].plot(tr['true'],label='True',lw=2)
        ax[i].plot(tr['pred'],label='Pred',lw=2)

        ax[i].set_ylim(0,1.6)
        ax[i].set_title(f'Trial {t}   R²={r2:.3f}')
        ax[i].grid(alpha=0.3)

    ax[0].legend()

    path='results/plots/s3_timeseries_gatsby.png'
    plt.tight_layout()
    plt.savefig(path,dpi=160)
    plt.close()

    print('saved →',path)


In [94]:
def plot_transfer_comparison():

    s1 = load_json(f'results/series1/history_{TARGET_RAT}_GRU.json')
    s3 = load_json('results/series3/history_finetune.json')

    best_s1 = max(s1['val_r2'])
    best_s3 = max(s3['val_r2'])

    plt.figure(figsize=(5,4))

    bars = plt.bar(
        ['Series1\nIndependent',
         'Series3\nTransfer'],
        [best_s1,best_s3]
    )

    for b,v in zip(bars,[best_s1,best_s3]):
        plt.text(
            b.get_x()+b.get_width()/2,
            v+0.01,
            f'{v:.3f}',
            ha='center'
        )

    plt.ylabel('Best Validation R²')
    plt.title(f'Transfer Learning Effect — {TARGET_RAT}')
    plt.grid(axis='y',alpha=0.3)

    path='results/plots/s3_transfer_compare.png'

    plt.tight_layout()
    plt.savefig(path,dpi=160)
    plt.close()

    print('saved →',path)


In [95]:
def run_series3_visualization():

    print('\nVisualizing Series3 results...')

    plot_series3_training_curves()
    plot_series3_scatter()
    plot_series3_timeseries()
    plot_transfer_comparison()

    print('\nAll plots saved in results/plots/')


In [96]:
run_series3_visualization()


Visualizing Series3 results...
saved → results/plots/s3_training_curves.png
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
saved → results/plots/s3_scatter_gatsby.png
  [data\hippo_nn_gatsby.npz] trials=85 neurons=66 train=68 val=17
saved → results/plots/s3_timeseries_gatsby.png
saved → results/plots/s3_transfer_compare.png

All plots saved in results/plots/
